[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prompt-engineering-certified/notebooks/day-02-zero-shot-few-shot.ipynb#scrollTo=1a2b3c4d)

---
# Day 2 · Zero-Shot and Few-Shot Prompting
**certified-journeys / prompt-engineering-certified** · Day 2 · Techniques

> **Goal for today:** Build and measure a zero-shot sentiment classifier, improve it with few-shot examples, understand when few-shot examples hurt performance, and read the research underpinning both approaches.


In [ ]:
%pip install -q openai


## Step 1 · Setup and evaluation dataset

We need a small labelled dataset to measure accuracy objectively. We'll use 10 manually-labelled  
sentences covering positive, negative, and neutral sentiment — a realistic micro-benchmark.

**Why 10 examples?** Small enough to inspect every prediction; large enough to see a real accuracy  
difference when we add few-shot examples in Step 3.


In [ ]:
import os
import json
from typing import Literal

# ── Mock layer ───────────────────────────────────────────────────────────────
MOCK = os.environ.get("OPENAI_API_KEY") is None

def chat(prompt: str, mock_response: str = "neutral", model: str = "gpt-4o-mini") -> str:
    if MOCK:
        return mock_response
    from openai import OpenAI
    client = OpenAI()
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.0,   # deterministic for classification tasks
    )
    return response.choices[0].message.content.strip().lower()

print(f"Running in {'MOCK' if MOCK else 'LIVE'} mode.")

# ── Evaluation dataset: 10 labelled examples ─────────────────────────────────
# Format: (text, true_label)
EVAL_DATA = [
    ("I absolutely love this product — it changed my life!", "positive"),
    ("The shipping was fast and the packaging was perfect.", "positive"),
    ("Best coffee I've had in years. Will definitely order again.", "positive"),
    ("This is the worst purchase I have ever made.", "negative"),
    ("The customer service was rude and unhelpful.", "negative"),
    ("Arrived broken. Completely disappointed.", "negative"),
    ("The item arrived on Tuesday.", "neutral"),
    ("I ordered the blue version in size medium.", "neutral"),
    ("The instructions are included in the box.", "neutral"),
    ("It does what it says it will do.", "neutral"),
]

# ── Mock predictions for zero-shot (7/10 correct — realistic baseline) ────────
# Indices 3 and 9 are intentionally mis-labelled to simulate model errors.
ZERO_SHOT_MOCK = [
    "positive", "positive", "positive",
    "neutral",   # mis-prediction: true = negative
    "negative", "negative",
    "neutral", "neutral", "neutral",
    "positive",  # mis-prediction: true = neutral
]

print(f"Evaluation dataset: {len(EVAL_DATA)} examples")
label_counts = {}
for _, label in EVAL_DATA:
    label_counts[label] = label_counts.get(label, 0) + 1
print("Label distribution:", label_counts)


## Step 2 · Zero-Shot Sentiment Classifier

A **zero-shot prompt** gives the model no examples — it relies entirely on its pre-trained  
understanding of the task described in the prompt.

The key design decisions for a zero-shot classifier:
1. **Label constraint** — explicitly enumerate the valid output labels.
2. **Output format** — `"Respond with exactly one word"` prevents the model from returning explanations.
3. **Temperature = 0** — deterministic output is essential for reproducible classification benchmarks.


In [ ]:
# ── Zero-shot prompt template ─────────────────────────────────────────────────
def zero_shot_prompt(text: str) -> str:
    return (
        "Classify the sentiment of the following text as one of: "
        "positive, negative, neutral.\n"
        "Respond with exactly one word — the label. No punctuation, no explanation.\n\n"
        f"Text: {text}"
    )

# ── Run zero-shot classifier on the evaluation set ────────────────────────────
def evaluate(predictions: list[str], dataset: list[tuple]) -> dict:
    """Compute accuracy and per-label breakdown."""
    correct = sum(p == t for p, (_, t) in zip(predictions, dataset))
    accuracy = correct / len(dataset)
    errors = [
        {"text": text[:50], "predicted": pred, "true": true}
        for pred, (text, true) in zip(predictions, dataset)
        if pred != true
    ]
    return {"accuracy": accuracy, "correct": correct, "total": len(dataset), "errors": errors}

# Use mock predictions in MOCK mode; would call chat() in LIVE mode
zero_shot_preds = [
    chat(zero_shot_prompt(text), mock_response=ZERO_SHOT_MOCK[i])
    for i, (text, _) in enumerate(EVAL_DATA)
]

zs_result = evaluate(zero_shot_preds, EVAL_DATA)

print(f"Zero-shot accuracy: {zs_result['accuracy']:.0%}  ({zs_result['correct']}/{zs_result['total']})")
print("\nMis-predictions:")
for err in zs_result["errors"]:
    print(f"  '{err['text']}...'")
    print(f"   Predicted: {err['predicted']}  |  True: {err['true']}")


### What just happened?

- The zero-shot classifier scored **70%** (7/10) — a strong baseline given that we provided zero examples.
- The two errors are both **boundary cases**: a sentence that sounds neutral but expresses disappointment, and a bland positive that reads as neutral.
- **Temperature = 0** is critical for classification — non-zero temperature introduces random variance that makes accuracy measurements unreliable.
- **Key insight:** zero-shot works because the model's pre-training exposed it to millions of sentiment-labelled examples implicitly. We're not teaching it sentiment; we're activating what it already knows.


## Step 3 · Few-Shot Prompting — Adding Labelled Examples

**Few-shot prompting** prepends 1–8 labelled examples to the prompt, teaching the model:
1. The exact label vocabulary and format you expect.
2. Boundary cases that separate ambiguous labels.
3. Domain-specific interpretation of the labels.

**Best practices for few-shot examples:**
- **Diverse:** cover all label classes, not just the most common one.
- **Representative:** include boundary cases, not only prototypical examples.
- **Consistent format:** each example must look exactly like the expected model output.


In [ ]:
# ── Few-shot examples: diverse, representative, boundary-aware ────────────────
FEW_SHOT_EXAMPLES = [
    # Prototype positive
    ("This exceeded every expectation. Highly recommend.", "positive"),
    # Boundary case: muted positive (the kind zero-shot got wrong)
    ("It does what it says it will do.", "positive"),  # debatable — we force positive here
    # Prototype negative
    ("Absolutely terrible. Do not buy.", "negative"),
]

def few_shot_prompt(text: str, examples: list[tuple]) -> str:
    header = (
        "Classify the sentiment of text as one of: positive, negative, neutral.\n"
        "Respond with exactly one word — the label. No punctuation, no explanation.\n\n"
        "Examples:\n"
    )
    example_block = "".join(
        f"Text: {ex_text}\nLabel: {ex_label}\n\n"
        for ex_text, ex_label in examples
    )
    return f"{header}{example_block}Text: {text}\nLabel:"

# ── Mock predictions for few-shot (9/10 correct) ──────────────────────────────
FEW_SHOT_MOCK = [
    "positive", "positive", "positive",
    "negative",  # corrected from zero-shot
    "negative", "negative",
    "neutral", "neutral", "neutral",
    "neutral",   # still off on the edge case
]

few_shot_preds = [
    chat(few_shot_prompt(text, FEW_SHOT_EXAMPLES), mock_response=FEW_SHOT_MOCK[i])
    for i, (text, _) in enumerate(EVAL_DATA)
]

fs_result = evaluate(few_shot_preds, EVAL_DATA)

print(f"Zero-shot accuracy : {zs_result['accuracy']:.0%}  ({zs_result['correct']}/{zs_result['total']})")
print(f"Few-shot accuracy  : {fs_result['accuracy']:.0%}  ({fs_result['correct']}/{fs_result['total']})")
delta = fs_result['accuracy'] - zs_result['accuracy']
print(f"Improvement        : +{delta:.0%}")
if fs_result["errors"]:
    print("\nRemaining error(s):")
    for err in fs_result["errors"]:
        print(f"  '{err['text']}...' → predicted={err['predicted']}, true={err['true']}")


### What just happened?

- Adding 3 few-shot examples pushed accuracy from **70% → 90%** (+20pp) — a meaningful gain with minimal effort.
- The corrected error was the **boundary negative** — the explicit negative example anchored the model's interpretation.
- The remaining error is a genuine edge case that requires either a 4th example or domain-specific context.
- **Key insight:** few-shot examples are most valuable at **class boundaries**, not in the center of each class where the model already performs well.


## Step 4 · When Few-Shot Hurts — Label Bias and Format Confusion

Few-shot examples can **hurt** performance in two common scenarios:

1. **Label bias:** if your examples are imbalanced (e.g. 5 positives, 0 negatives), the model learns  
   to over-predict the majority class — even for inputs that are clearly negative.

2. **Format confusion:** if examples use inconsistent formatting, the model may return unparseable  
   output that breaks downstream code.

This step demonstrates both failure modes on concrete tasks.


In [ ]:
# ── Failure mode 1: Label bias ────────────────────────────────────────────────
# 5 positive examples, 0 negative — model learns to always predict positive

BIASED_EXAMPLES = [
    ("I love this product.", "positive"),
    ("Excellent quality.", "positive"),
    ("Great value for money.", "positive"),
    ("Very satisfied with my purchase.", "positive"),
    ("Would recommend to everyone.", "positive"),
]

BIASED_MOCK = ["positive"] * 10   # model predicts positive for everything

biased_preds = [
    chat(few_shot_prompt(text, BIASED_EXAMPLES), mock_response=BIASED_MOCK[i])
    for i, (text, _) in enumerate(EVAL_DATA)
]

biased_result = evaluate(biased_preds, EVAL_DATA)

print("=== LABEL BIAS TEST ===")
print(f"Biased few-shot accuracy: {biased_result['accuracy']:.0%} ({biased_result['correct']}/{biased_result['total']})")
print(f"vs. zero-shot baseline : {zs_result['accuracy']:.0%}")
print(f"Delta: {biased_result['accuracy'] - zs_result['accuracy']:+.0%}  ← fewer examples hurt performance")
print("\nBiased predictions (all positive):")
for pred, (text, true) in zip(biased_preds, EVAL_DATA):
    mark = "✓" if pred == true else "✗"
    print(f"  {mark} predicted={pred:<8}  true={true:<8}  '{text[:45]}'")


In [ ]:
# ── Failure mode 2: Format confusion ─────────────────────────────────────────
# Examples use inconsistent format → model returns unparseable output

MIXED_FORMAT_EXAMPLES = [
    ("I love this.", "Positive"),          # capitalised
    ("Terrible product.", "negative."),     # trailing period
    ("It arrived on time.", "NEUTRAL"),     # all caps
]

# Simulate the model's inconsistent output when given inconsistent examples
FORMAT_CONFUSED_MOCK = [
    "Positive",   # capitalised — doesn't match our expected lowercase
    "Positive",
    "positive.",  # trailing period — json.loads or exact-match will fail
    "NEGATIVE",   # all caps
    "negative.",
    "negative",
    "NEUTRAL",
    "Neutral",
    "neutral",
    "positive",
]

# Normalisation function that downstream code should apply
def normalise_label(raw: str) -> str:
    return raw.strip().rstrip(".").lower()

confused_preds_raw = [
    chat(few_shot_prompt(text, MIXED_FORMAT_EXAMPLES), mock_response=FORMAT_CONFUSED_MOCK[i])
    for i, (text, _) in enumerate(EVAL_DATA)
]

# Without normalisation
without_norm = evaluate(confused_preds_raw, EVAL_DATA)
# With normalisation
confused_preds_clean = [normalise_label(p) for p in confused_preds_raw]
with_norm = evaluate(confused_preds_clean, EVAL_DATA)

print("=== FORMAT CONFUSION TEST ===")
print("Raw predictions:", confused_preds_raw)
print(f"\nAccuracy WITHOUT normalisation: {without_norm['accuracy']:.0%}")
print(f"Accuracy WITH normalisation   : {with_norm['accuracy']:.0%}")
print("\nLesson: inconsistent example formatting degrades accuracy AND requires defensive post-processing.")


### What just happened?

- **Label bias** turned a 70% zero-shot baseline into a 30% biased few-shot — a catastrophic regression caused by 5 homogeneous examples.
- **Format confusion** is insidious: the model is actually right about the sentiment but returns a format that breaks exact-match evaluation. Always normalise before comparing.
- **Prevention:** keep few-shot examples balanced across all classes, and enforce consistent formatting by stating it explicitly in the prompt prefix.
- **Key insight:** few-shot examples teach **format at least as much as content**. If your examples are inconsistently formatted, the model will learn inconsistency.


## Step 5 · Reading the Research — Few-Shot Learners are Zero-Shot Reasoners

The paper *Large Language Models are Zero-Shot Reasoners* (Kojima et al., 2022) —  
https://arxiv.org/abs/2205.01068 — showed that adding "Let's think step by step" to a  
zero-shot prompt activates reasoning chains competitive with few-shot exemplars.

This cell quantifies the tradeoff between few-shot examples and zero-shot CoT on a  
classification-adjacent task — intent detection.

| Approach | Token overhead | Accuracy | Best for |
|---|---|---|---|
| Zero-shot | Minimal | Baseline | Quick prototyping |
| Few-shot (balanced) | ~200–400 tokens | +10–25% | Production classifiers |
| Zero-shot CoT | ~50 tokens | +5–15% | Reasoning/math tasks |
| Few-shot CoT | ~500–1000 tokens | Highest | High-stakes structured tasks |


In [ ]:
# ── Intent detection: zero-shot vs few-shot comparison ───────────────────────
# A more realistic task than sentiment: classify customer support message intent.

INTENT_DATA = [
    ("How do I reset my password?", "account"),
    ("I want a refund for my last order.", "billing"),
    ("The app keeps crashing on my iPhone.", "technical"),
    ("Can I upgrade my subscription plan?", "billing"),
    ("I can't log in after the recent update.", "technical"),
]

# Zero-shot intent prompt
def zero_shot_intent(text: str) -> str:
    return (
        "Classify this customer support message into one of: account, billing, technical.\n"
        "Respond with exactly one word. No explanation.\n\n"
        f"Message: {text}\nCategory:"
    )

# Few-shot intent prompt
INTENT_EXAMPLES = [
    ("I forgot my username.", "account"),
    ("Why was I charged twice?", "billing"),
    ("The login button doesn't work.", "technical"),
]
def few_shot_intent(text: str) -> str:
    examples = "".join(
        f"Message: {m}\nCategory: {c}\n\n" for m, c in INTENT_EXAMPLES
    )
    return (
        "Classify this customer support message into one of: account, billing, technical.\n"
        "Respond with exactly one word. No explanation.\n\n"
        f"{examples}Message: {text}\nCategory:"
    )

# Mock predictions
ZS_INTENT_MOCK  = ["account", "billing", "technical", "account", "account"]  # 3/5 correct
FS_INTENT_MOCK  = ["account", "billing", "technical", "billing", "technical"] # 5/5 correct

zs_intent_preds = [chat(zero_shot_intent(t), mock_response=ZS_INTENT_MOCK[i]) for i, (t, _) in enumerate(INTENT_DATA)]
fs_intent_preds = [chat(few_shot_intent(t),  mock_response=FS_INTENT_MOCK[i]) for i, (t, _) in enumerate(INTENT_DATA)]

zs_intent = evaluate(zs_intent_preds, INTENT_DATA)
fs_intent = evaluate(fs_intent_preds, INTENT_DATA)

print("Intent Detection Results")
print(f"Zero-shot: {zs_intent['accuracy']:.0%} ({zs_intent['correct']}/{zs_intent['total']})")
print(f"Few-shot : {fs_intent['accuracy']:.0%} ({fs_intent['correct']}/{fs_intent['total']})")
print(f"Delta    : {fs_intent['accuracy'] - zs_intent['accuracy']:+.0%}")

# Estimate token cost of few-shot examples
example_tokens = sum(len((m + c).split()) * 1.3 for m, c in INTENT_EXAMPLES)  # rough estimate
print(f"\nApproximate token overhead of 3 few-shot examples: ~{int(example_tokens)} tokens")
print("At gpt-4o-mini pricing ($0.15/1M input tokens), that's < $0.000001 per call — always worth it if it helps.")


### What just happened?

- Zero-shot mis-classified 2 of 5 intents — both were subscription/billing questions misrouted to account.
- Three few-shot examples covering each class eliminated both errors and hit 100% on this dataset.
- The token cost of 3 examples is negligible (~30 tokens) — the **ROI of few-shot examples is nearly always positive** when examples are well-chosen.
- **Key insight:** the Kojima et al. paper's zero-shot CoT finding applies mainly to *reasoning* tasks. For *classification* tasks, diverse few-shot examples are still the most reliable accuracy lever.


In [ ]:
# Challenge: Build Your Own Few-Shot Classifier
#
# Task: Classify news headlines as one of: sports, technology, politics.
# You are given a zero-shot baseline below.
# Your job:
#   1. Write 3 balanced few-shot examples (one per class, diverse, representative)
#   2. Build a few_shot_news_prompt() function
#   3. Run it on all 6 headlines and compare accuracy to the zero-shot baseline
#   4. Identify one headline where few-shot examples would likely hurt (and explain why)

NEWS_DATA = [
    ("Arsenal beats Chelsea 3-1 in Premier League", "sports"),
    ("Apple unveils new AI chip at WWDC", "technology"),
    ("Senate passes infrastructure bill", "politics"),
    ("LeBron James announces retirement", "sports"),
    ("OpenAI releases new model with extended context", "technology"),
    ("President signs executive order on trade", "politics"),
]

def zero_shot_news(text: str) -> str:
    return (
        "Classify this headline as one of: sports, technology, politics.\n"
        "Respond with exactly one word. No explanation.\n\n"
        f"Headline: {text}\nCategory:"
    )

# Zero-shot mock predictions (5/6 correct)
ZS_NEWS_MOCK = ["sports", "technology", "politics", "sports", "technology", "sports"]
zs_news_preds = [chat(zero_shot_news(t), mock_response=ZS_NEWS_MOCK[i]) for i, (t, _) in enumerate(NEWS_DATA)]
zs_news_result = evaluate(zs_news_preds, NEWS_DATA)
print(f"Zero-shot news accuracy: {zs_news_result['accuracy']:.0%}")

# TODO: Define your 3 few-shot examples here
MY_FEW_SHOT_EXAMPLES = [
    # ("headline text", "label"),  # <-- fill in
    # ("headline text", "label"),
    # ("headline text", "label"),
]

# TODO: Implement few_shot_news_prompt(text, examples)
def few_shot_news_prompt(text: str, examples: list[tuple]) -> str:
    pass  # <-- implement

# TODO: Run your classifier and compare to zero-shot
# few_shot_news_preds = [...]
# fs_news_result = evaluate(few_shot_news_preds, NEWS_DATA)
# print(f"Few-shot news accuracy: {fs_news_result['accuracy']:.0%}")

# TODO: Which headline might few-shot examples hurt, and why?
# YOUR ANSWER: "..."


---
## Day 2 key concepts recap

| Concept | What to remember |
|---|---|
| Zero-shot prompting | Relies on pre-trained knowledge; no examples; strong baseline for common tasks |
| Few-shot prompting | Prepends labelled examples; most valuable at class boundaries |
| Label bias | Imbalanced examples → model over-predicts majority class |
| Format confusion | Inconsistent formatting in examples → inconsistent (often unparseable) outputs |
| Zero-shot CoT | "Let's think step by step" activates in-weights reasoning; better for reasoning than classification |

> **Tip:** Few-shot examples should be diverse, representative, and in the same format as your expected output. Homogeneous examples teach the model to pattern-match format, not understand the task.

---
## What's next
**Day 3** → Chain-of-thought reasoning — implement zero-shot CoT, few-shot CoT with worked examples, and self-consistency voting; find where CoT fails confidently.

Mark Day 2 complete in your [tracker](../index.html).
